# Qwen3.5-0.8B + FlyEmbedding-v3.3
## Progressive full vocabulary replacement

This experiment progressively removes Qwen's full vocabulary representation:

    beta=0.00  -> 100% Qwen vocabulary
    beta=0.05
    beta=0.10
    beta=0.20
    beta=0.35
    beta=0.50
    beta=0.70
    beta=0.85
    beta=1.00  -> 100% Compact Fly vocabulary

Both the **input embedding and output vocabulary head** transition together to the compact Fly core.

At beta=1, the compact candidate no longer uses the full Qwen vocabulary matrix. The notebook reports the actual total parameter reduction instead of assuming it reaches 50%.

A beta=1 model is always attempted and saved, but quality must be demonstrated by the validation and generation gates.


In [ ]:
#@title 1. Setup
import pathlib, subprocess, sys, importlib, torch, json, shutil, os
REPO_DIR=pathlib.Path('/content/TinyCeNN-LM')

if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)],check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q','-U',
    'transformers','accelerate','datasets','huggingface_hub',
    'safetensors','pandas','matplotlib','lm-eval','tabulate'
],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)

SRC_DIR=REPO_DIR/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
for name in list(sys.modules):
    if name=='tinycenn_lm' or name.startswith('tinycenn_lm.'):
        del sys.modules[name]
importlib.invalidate_caches()

for p in [
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_flyembedding_v33.py',
    REPO_DIR/'scripts'/'run_qwen35_flyembedding_v33.py'
]:
    subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)

print('✓ FlyEmbedding-v3.3 preflight OK')
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
else:
    print('⚠️ Switch Colab to GPU runtime.')


In [ ]:
#@title 2. Configuration
BASE_MODEL='Qwen/Qwen3.5-0.8B' #@param {type:'string'}
RUN_MODE='quick' #@param ['quick','strong']
SEQ_LEN=128 #@param {type:'integer'}

LATENT_DIM=256 #@param [64,128,192,256,384,512] {type:'raw'}
FLY_NODES=256 #@param {type:'integer'}
GRAPH_STEPS=1 #@param {type:'integer'}
GRAPH_MIX=0.05 #@param {type:'number'}
MAX_FLY_SCALE=0.05 #@param {type:'number'}

LR=0.00020 #@param {type:'number'}
STAGE_UPDATES=100 #@param {type:'integer'}
PROBE_EVERY=20 #@param {type:'integer'}
EXTEND_UPDATES=100 #@param {type:'integer'}
MAX_STAGE_UPDATES=1200 #@param {type:'integer'}
PATIENCE_PROBES=12 #@param {type:'integer'}

MIN_TOP1=0.97 #@param {type:'number'}
MAX_KL=0.03 #@param {type:'number'}
MAX_CE_GAP=0.08 #@param {type:'number'}

RUN_FAST_EVAL=True #@param {type:'boolean'}
FAST_EVAL_LIMIT=50 #@param {type:'integer'}

OUTPUT_DIR=REPO_DIR/'results'/'flyembedding_v33_qwen35_08b'
EVAL_MODEL_DIR=REPO_DIR/'results'/'flyembedding_v33_materialized_eval'
EVAL_DIR=REPO_DIR/'results'/'flyembedding_v33_eval'

print('Latent dim:',LATENT_DIM)
print('Progressive schedule: 0 -> .05 -> .10 -> .20 -> .35 -> .50 -> .70 -> .85 -> 1.0')
print('Each beta restores its own best validation checkpoint before moving on.')
print('Base updates:',STAGE_UPDATES,'extend:',EXTEND_UPDATES,'max/stage:',MAX_STAGE_UPDATES)
print('Important: beta=1 is full compact vocabulary replacement.')


In [ ]:
#@title 3. Train progressive FlyEmbedding-v3.3
cmd=[
    sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_flyembedding_v33.py'),
    '--base-model',BASE_MODEL,
    '--run-mode',RUN_MODE,
    '--seq-len',str(SEQ_LEN),
    '--latent-dim',str(LATENT_DIM),
    '--fly-nodes',str(FLY_NODES),
    '--graph-steps',str(GRAPH_STEPS),
    '--graph-mix',str(GRAPH_MIX),
    '--max-fly-scale',str(MAX_FLY_SCALE),
    '--lr',str(LR),
    '--stage-updates',str(STAGE_UPDATES),
    '--probe-every',str(PROBE_EVERY),
    '--extend-updates',str(EXTEND_UPDATES),
    '--max-stage-updates',str(MAX_STAGE_UPDATES),
    '--patience-probes',str(PATIENCE_PROBES),
    '--min-top1',str(MIN_TOP1),
    '--max-kl',str(MAX_KL),
    '--max-ce-gap',str(MAX_CE_GAP),
    '--output-dir',str(OUTPUT_DIR)
]
print('='*110); print(' '.join(cmd)); print('='*110)
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in iter(p.stdout.readline,''):
    print(line,end='',flush=True)
rc=p.wait()
print('\nFinished, exit code',rc)
if rc: raise subprocess.CalledProcessError(rc,cmd)

report=json.loads((OUTPUT_DIR/'report.json').read_text())
print('\nBEST SAFE BETA:',report['best_safe_beta'])
print('BETA=1 QUALITY GATE:',report['beta1_quality_gate_passed'])
print('ACTUAL TOTAL PARAM REDUCTION:',round(report['actual_total_param_reduction_pct'],2),'%')
print('TARGET HALF SIZE REACHED:',report['target_half_size_reached'])
print('Compact vocab stats:',json.dumps(report['compact_vocab_stats'],indent=2))


In [ ]:
#@title 4. Progressive replacement curves
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

stages=pd.read_csv(OUTPUT_DIR/'stage_validation.csv')
hist=pd.read_csv(OUTPUT_DIR/'training_history.csv')
display(stages[['beta','initial_student_ce','best_step','trained_steps','student_ce','ce_gap','teacher_kl','top1_logit_agreement','safe','violation_score']])

plt.figure(figsize=(9,5))
plt.plot(stages['beta'],stages['top1_logit_agreement'],marker='o')
plt.axhline(MIN_TOP1,linestyle='--')
plt.xlabel('beta: fraction Compact Fly vocabulary')
plt.ylabel('Top-1 agreement with Qwen')
plt.title('Quality preservation during Qwen vocabulary removal')
plt.show()

plt.figure(figsize=(9,5))
plt.plot(stages['beta'],stages['student_ce'],marker='o',label='Fly-v3.3')
plt.plot(stages['beta'],stages['teacher_ce'],marker='o',label='Qwen')
plt.xlabel('beta')
plt.ylabel('Validation CE')
plt.legend()
plt.show()

plt.figure(figsize=(9,5))
plt.plot(stages['beta'],stages['teacher_kl'],marker='o')
plt.axhline(MAX_KL,linestyle='--')
plt.xlabel('beta')
plt.ylabel('KL vs Qwen')
plt.title('Distribution drift')
plt.show()


In [ ]:
#@title 5. Inspect beta=1 full Fly candidate
for i,x in enumerate(report['beta1_generation'],1):
    print('\n'+'='*100)
    print(i,'USER:',x['prompt'])
    print('\nQWEN:',x['qwen'])
    print('\nFLY beta=1:',x['fly'])
    print('\nexact=',x['exact'],'| jaccard=',round(x['jaccard'],3),'| valid=',x['valid'])

print('\nBETA=1 probe:')
print(json.dumps(report['beta1_probe'],indent=2))
print('\nCompact-only unique params:',f"{report['compact_only_unique_params']:,}")
print('Original unique params:',f"{report['original_unique_params']:,}")
print('Actual total parameter reduction:',f"{report['actual_total_param_reduction_pct']:.2f}%")


In [ ]:
#@title 6. Rebuild beta=1 compact model and materialize temporarily for standard FastEval
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.qwen35_flyembedding_v33 import (
    FlyEmbeddingV33Config, factorize_vocab_weight_v33,
    install_fly_embedding_v33, set_progressive_beta_v33
)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=torch.bfloat16 if device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type=='cuda' else torch.float32)
tok=AutoTokenizer.from_pretrained(BASE_MODEL,use_fast=True)
if tok.pad_token_id is None: tok.pad_token=tok.eos_token

m=AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype,low_cpu_mem_usage=True).to(device).eval()
# Shapes only are needed for installation; beta1 state overwrites the factorized initialization.
fac=factorize_vocab_weight_v33(m.model.embed_tokens.weight,LATENT_DIM)
cfg=FlyEmbeddingV33Config(
    latent_dim=LATENT_DIM,fly_nodes=FLY_NODES,graph_steps=GRAPH_STEPS,
    graph_mix=GRAPH_MIX,max_fly_scale=MAX_FLY_SCALE
)
install_fly_embedding_v33(m,cfg,fac)
beta1=torch.load(OUTPUT_DIR/'fly_v33_beta1_compact.pt',map_location='cpu')
m.fly_embedding_v33_core.load_state_dict(beta1['core'],strict=True)
set_progressive_beta_v33(m,1.0)
m.eval()

# Materialize only for lm-eval compatibility. This temporary model is NOT the compact storage format.
core=m.fly_embedding_v33_core
with torch.no_grad():
    in_weight=core.effective_weight(chunk_rows=4096)
    out_weight=torch.matmul(core.codebook.weight.detach().cpu(),core.basis.detach().cpu())

hidden=int(m.config.hidden_size); vocab=int(m.config.vocab_size)
new_emb=torch.nn.Embedding(vocab,hidden,device=device,dtype=dtype)
new_head=torch.nn.Linear(hidden,vocab,bias=False,device=device,dtype=dtype)
new_emb.weight.copy_(in_weight.to(device=device,dtype=dtype))
new_head.weight.copy_(out_weight.to(device=device,dtype=dtype))
m.model.embed_tokens=new_emb
m.lm_head=new_head
m.config.tie_word_embeddings=False
if hasattr(m,'fly_embedding_v33_core'): delattr(m,'fly_embedding_v33_core')

if EVAL_MODEL_DIR.exists(): shutil.rmtree(EVAL_MODEL_DIR)
EVAL_MODEL_DIR.mkdir(parents=True,exist_ok=True)
m.save_pretrained(EVAL_MODEL_DIR,safe_serialization=True,max_shard_size='2GB')
tok.save_pretrained(EVAL_MODEL_DIR)
print('✓ Temporary materialized eval model saved:',EVAL_MODEL_DIR)
print('NOTE: this directory is larger than the real compact-only model because it materializes both vocab matrices.')

del m
if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
#@title 7. FastEval beta=1 vs original Qwen
if RUN_FAST_EVAL:
    TASKS='hellaswag,piqa,arc_easy,arc_challenge,winogrande,gsm8k'
    if EVAL_DIR.exists(): shutil.rmtree(EVAL_DIR)
    EVAL_DIR.mkdir(parents=True,exist_ok=True)

    def run_eval(label,pretrained):
        out=EVAL_DIR/label; out.mkdir(parents=True,exist_ok=True)
        dtype_name='bfloat16' if dtype==torch.bfloat16 else ('float16' if dtype==torch.float16 else 'float32')
        cmd=[
            sys.executable,'-m','lm_eval','--model','hf',
            '--model_args',f'pretrained={pretrained},dtype={dtype_name},trust_remote_code=True',
            '--tasks',TASKS,'--batch_size','auto','--limit',str(FAST_EVAL_LIMIT),
            '--output_path',str(out)
        ]
        print('\n',label,':',' '.join(cmd))
        subprocess.run(cmd,check=False)

    run_eval('fly_v33_beta1',str(EVAL_MODEL_DIR))
    run_eval('qwen_base',BASE_MODEL)

    def newest(folder):
        xs=[]
        for p in pathlib.Path(folder).rglob('*.json'):
            try:
                d=json.loads(p.read_text())
                if isinstance(d,dict) and 'results' in d: xs.append((p.stat().st_mtime,d))
            except: pass
        return max(xs,key=lambda x:x[0])[1] if xs else None

    fly_eval=newest(EVAL_DIR/'fly_v33_beta1')
    base_eval=newest(EVAL_DIR/'qwen_base')

    def metric(task,d):
        r=(d or {}).get('results',{}).get(task,{})
        for k in ['acc_norm,none','acc,none','exact_match,strict-match','exact_match,flexible-extract']:
            if k in r:return float(r[k]),k
        return None,None

    rows=[]
    for task in TASKS.split(','):
        fv,fk=metric(task,fly_eval); bv,bk=metric(task,base_eval)
        rows.append({'task':task,'metric':fk or bk,'Qwen':bv,'Fly-v3.3 beta=1':fv,
                     'delta':(fv-bv) if fv is not None and bv is not None else None})
    bench_df=pd.DataFrame(rows)
    display(bench_df)
    bench_df.to_csv(OUTPUT_DIR/'fast_eval_beta1.csv',index=False)
else:
    print('RUN_FAST_EVAL=False')


## How to read v3.3

The critical outputs are:

- **best_safe_beta** — how far Qwen's vocabulary could be removed while staying inside the quality boundary.
- **beta1_quality_gate_passed** — whether the fully compact Fly vocabulary actually preserved acceptable validation quality.
- **actual_total_param_reduction_pct** — the real reduction after removing the full Qwen vocabulary matrix.
- **target_half_size_reached** — whether vocabulary replacement alone truly reached 50% fewer total parameters.

Do not treat beta=1 as successful merely because it runs. It is successful only if validation, generation, and FastEval stay close to Qwen.

If beta=1 fails but beta=0.5 or 0.7 succeeds, that is still a useful result: v3.4 can focus on improving the compact representation before attempting complete replacement again.
